# BE-HPG — Phase 5: Boundary-loss ablation (lambda)

Sweeps the boundary-loss weight **lambda in {0, 0.1, 0.3, 0.5}** for BE-HPG on Spleen
(lambda=0 = no boundary loss). Cheaper budget (~1000 episodes/run). Same setup otherwise.

Run with `SMOKE=True` once to verify, then `SMOKE=False` for the real ablation
(~4 x 8–12 min ≈ 30–45 min total, resumable).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/be-hpg'
print('Drive:', DRIVE_ROOT)

In [ ]:
!pip -q install timm medpy

In [ ]:
import os, sys, shutil
FORCE_UPLOAD = False
zip_drive = f'{DRIVE_ROOT}/be-hpg-src.zip'
if FORCE_UPLOAD or not os.path.exists(zip_drive):
    from google.colab import files
    up = files.upload()
    with open(zip_drive, 'wb') as f:
        f.write(up[list(up)[0]])
code_dir = '/content/be-hpg'
shutil.rmtree(code_dir, ignore_errors=True); os.makedirs(code_dir, exist_ok=True)
os.system(f'unzip -q -o "{zip_drive}" -d "{code_dir}"')
if code_dir not in sys.path:
    sys.path.insert(0, code_dir)
for _m in [k for k in sys.modules if k == 'src' or k.startswith('src.')]:
    del sys.modules[_m]        # drop cached modules so a re-uploaded zip takes effect (no restart needed)
import inspect
from src.models.be_hpg import BEHPG
print('code ready | new BE-HPG loaded =', 'use_ssp' in inspect.signature(BEHPG.__init__).parameters)

In [ ]:
import torch
SMOKE = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
suffix = '_smoke' if SMOKE else ''
NPZ = f'{DRIVE_ROOT}/data/spleen/spleen{suffix}.npz'
MANIFEST = f'{DRIVE_ROOT}/data/spleen/manifest{suffix}.csv'
TRAIN_EP = 40 if SMOKE else 1000
EVAL_EP = 10 if SMOKE else 100
EPISODE_SEED = 1234
LAMBDAS = [0.0, 0.1, 0.3, 0.5]
print('device:', DEVICE, '| SMOKE', SMOKE, '| lambdas', LAMBDAS)

In [ ]:
import time, json
import pandas as pd
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from src.utils.config import load_config
from src.utils.seed import set_seed
from src.engine.build import build_model
from src.engine.train import train_episodes
from src.engine.eval import eval_episodes
from src.engine.checkpoint import load_checkpoint
from src.data.sampler import sampler_from_npz

mdf = pd.read_csv(MANIFEST) if os.path.exists(MANIFEST) else None
TRAIN_SPLIT = None if SMOKE else 'train'
TEST_SPLIT = None if SMOKE else 'test'
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)

In [ ]:
ablation = {}
for lam in LAMBDAS:
    cfg = load_config(f'{code_dir}/configs/be_hpg.yaml', [f'loss.boundary_lambda={lam}'])
    set_seed(cfg.get('seed', 42))
    tag = f'be_hpg_lam{lam}'
    print('=' * 60); print('lambda =', lam)
    model = build_model(cfg)
    tr = sampler_from_npz(NPZ, mdf, TRAIN_SPLIT, k_shots=cfg['episode']['k_shot_train'],
                          query_size=cfg['episode']['query_size'], seed=cfg.get('seed', 42))
    opt = AdamW(model.parameters(), lr=cfg['train']['lr'], weight_decay=cfg['train']['weight_decay'])
    sched = CosineAnnealingLR(opt, T_max=TRAIN_EP)
    ckpt = f'{DRIVE_ROOT}/checkpoints/{tag}_spleen{suffix}.pt'
    start = 0
    if (not SMOKE) and os.path.exists(ckpt):
        start, _ = load_checkpoint(ckpt, model, opt, sched, map_location=DEVICE)
    t0 = time.time()
    if start < TRAIN_EP:
        train_episodes(model, tr, episodes=TRAIN_EP, start_episode=start, optimizer=opt, scheduler=sched,
                       lam=lam, device=DEVICE, amp=cfg.get('amp', True), ckpt_path=ckpt, ckpt_every=250,
                       log_every=max(TRAIN_EP // 5, 1), balanced=cfg['loss'].get('balanced_bce', True),
                       max_pw=cfg['loss'].get('bce_max_pos_weight', 20.0))
    res = {'lambda': lam, 'train_episodes': TRAIN_EP, 'by_shot': {}}
    for k in (1, 5):
        ev = sampler_from_npz(NPZ, mdf, TEST_SPLIT, k_shots=[k],
                              query_size=cfg['episode']['query_size'], seed=EPISODE_SEED)
        res['by_shot'][f'{k}shot'] = eval_episodes(model, ev, episodes=EVAL_EP, k_shot=k, device=DEVICE)
    json.dump(res, open(f'{DRIVE_ROOT}/results/{tag}_spleen{suffix}.json', 'w'), indent=2)
    ablation[lam] = res
    print(f'  done in {time.time() - t0:.0f}s | 1-shot dice {res["by_shot"]["1shot"]["dice"]:.3f} '
          f'| 5-shot dice {res["by_shot"]["5shot"]["dice"]:.3f}')

print('\n==== lambda ablation (1-shot / 5-shot Dice) ====')
for lam, r in ablation.items():
    print(f'  lambda={lam}: {r["by_shot"]["1shot"]["dice"]:.3f} / {r["by_shot"]["5shot"]["dice"]:.3f}')

## Report back
Paste the lambda-ablation Dice summary, and download the 4 `results/be_hpg_lam*_spleen.json`
into the repo `results/` folder. I'll build the ablation table for the paper.